# Tasks 6–7: Dashboard and Genie

## Dashboard

I created a Databricks dashboard using the Gold layer tables and views.

Objects used:

- `marathos.gold.fct_results`
- `marathos.gold.dim_event`
- `marathos.gold.dim_athlete`
- `marathos.gold.dim_date`
- `marathos.gold.v_marathon_km_results`
- `marathos.gold.v_marathon_mi_results`

The dashboard includes:

- Total results
- Total athletes
- Results by year
- Top athlete countries
- Athlete gender distribution
- KM vs MI results

The dashboard was created with help from Genie Code and reviewed against the Gold tables.

The final dashboard values included:

```text
Total Results = 509,763
Total Athletes = 1,387,264
```

The dashboard helps users explore marathon participation, country distribution, yearly trends, athlete gender distribution, and distance-unit comparisons.

---

## Genie

I created a Genie Space connected to the Gold layer.

The lecturer required at least two Genie tests. For each test, I first checked the answer manually with SQL and then compared it with Genie.

Some Genie answers matched the manual SQL checks, while some did not. I treated the manual SQL results as the source of truth.

---

### Genie Test 1: Results by event year

**Question asked to Genie:**

Using `marathos.gold.fct_results` joined with `marathos.gold.dim_event` on `event_id`, how many results are there by `event_year`? Order by `event_year`.

**Manual SQL check:**

```sql
SELECT
  e.event_year,
  COUNT(*) AS result_count
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e
  ON f.event_id = e.event_id
GROUP BY e.event_year
ORDER BY e.event_year;
```

**Genie result:**

Genie returned yearly result counts from 2010 to 2022. The result showed growth from 2010 to 2019, a sharp drop in 2020, and recovery in 2021 and 2022.

Example Genie result:

```text
2010 = 187,887
2011 = 211,367
2012 = 269,737
2013 = 316,846
2014 = 384,196
2015 = 441,748
2016 = 500,937
2017 = 558,921
2018 = 613,443
2019 = 675,398
2020 = 204,723
2021 = 338,946
2022 = 425,021
```

**Evaluation:**

This matched the dashboard trend and the manual SQL check, so I accepted this Genie answer.

This shows that Genie can answer trend questions when the question clearly points to the Gold fact table and event dimension.

---

### Genie Test 2: Average finish time for USA in 2018

**Question asked to Genie:**

What was the average finish time in hours for athletes from USA in 2018?

**Manual SQL check:**

```sql
SELECT
  a.athlete_country,
  e.event_year,
  ROUND(AVG(f.performance_seconds) / 3600, 2) AS avg_finish_hours
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
JOIN marathos.gold.dim_event e
  ON f.event_id = e.event_id
WHERE a.athlete_country = 'USA'
  AND e.event_year = 2018
GROUP BY a.athlete_country, e.event_year;
```

**Manual result:**

```text
12.02 hours
```

**Genie result:**

Genie returned:

```text
12.02 hours
```

**Evaluation:**

This matched the manual SQL result, so I accepted this Genie answer.

This shows that Genie can answer more specific analytical questions when the Gold layer is properly modelled and the question is clear.

---

### Genie Test 3: Average speed by athlete gender

**Question asked to Genie:**

Using `marathos.gold.fct_results` joined with `marathos.gold.dim_athlete` on `athlete_id`, what is the average `speed_kmh` by `athlete_gender`?

**Manual SQL check:**

```sql
SELECT
  a.athlete_gender,
  ROUND(AVG(f.speed_kmh), 2) AS avg_speed_kmh
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_gender
ORDER BY avg_speed_kmh DESC;
```

**Manual result:**

```text
X = 7.36
M = 7.32
F = 7.02
```

**Genie result:**

Genie returned the same values:

```text
X = 7.36
M = 7.32
F = 7.02
```

**Evaluation:**

This matched the manual SQL result, so I accepted this Genie answer.

This shows that Genie can answer grouped aggregation questions when the join between the fact table and dimension table is clearly stated.

---

### Genie Test 4: Total race results

**Question asked to Genie:**

How many race results are in the final Gold fact table?

**Manual SQL check:**

```sql
SELECT COUNT(*) AS total_results
FROM marathos.gold.fct_results;
```

**Manual result:**

```text
509,763
```

**Genie result:**

Genie returned:

```text
5,097,363
```

**Evaluation:**

This did not match the manual SQL result. I treated the manual SQL result as the source of truth because it directly queried `marathos.gold.fct_results`.

I also tried to correct Genie by adding instructions that `marathos.gold.fct_results` should be used as the source of truth for final cleaned race results. However, Genie still returned a different result.

This shows that Genie can make mistakes, so I verified the result manually before accepting it.

---

### Genie Test 5: Top athlete countries

**Question asked to Genie:**

Which athlete countries have the most results?

**Manual SQL check:**

```sql
SELECT
  a.athlete_country,
  COUNT(*) AS result_count
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a
  ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_country
ORDER BY result_count DESC
LIMIT 10;
```

**Manual result:**

The manual SQL result matched the dashboard chart for top athlete countries. The leading countries included:

```text
FRA
JPN
GBR
ITA
GER
```

**Genie result:**

Genie returned a different country ranking, including USA and France as the leading countries.

**Evaluation:**

This did not match my manual SQL result or the dashboard chart.

I treated the manual SQL query and dashboard as the source of truth. This confirmed that Genie answers must be reviewed before being accepted.

---

## Notes

The dashboard and Genie use the cleaned Silver OBT and the Gold star schema.

The purpose is to help users explore:

- marathon participation
- country distribution
- yearly trends
- athlete gender distribution
- race type differences
- distance unit comparisons

Genie was useful for asking business questions about the Gold layer and dashboard. However, the answers were verified with manual SQL because AI-generated answers can sometimes contain mistakes.

In this project, I accepted Genie answers when they matched manual SQL checks, and I rejected Genie answers when they did not match the validated Gold tables or dashboard.